# 2.5 Reinforcement Learning — Agent Routing Optimizer — CLO05 (RL & Decision Systems)

**Project M.O.N.D.A.Y** · Course 24CSE0316 · Reinforcement Learning (Bandit & Decision Systems).

**Goal.** Dynamically learn optimal routing strategies for task execution using an $\epsilon$-Greedy Multi-Armed Bandit model to balance exploration and exploitation.

**CLO05 evidence produced here:** Cumulative Reward Curves over time vs Fixed/Random Baselines, Regret Curves, and the required Supervised vs Unsupervised vs RL Paradigm Comparison Report.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ml.rl_router import (
    EpsilonGreedyBandit,
    simulate_environment_step,
    ROUTING_ACTIONS,
    OPTIMAL_REWARD,
    OPTIMAL_ACTION
)

## 1. Environment Simulation & Bandit Setup

The agent selects among 4 routing strategies: `direct_dispatch`, `planner_routed`, `hybrid_check`, and `clarify_user`.

In [ ]:
n_steps = 1000
bandit = EpsilonGreedyBandit(n_arms=len(ROUTING_ACTIONS), epsilon=0.3, epsilon_decay=0.995, seed=42)

rewards_bandit = []
rewards_random = []
rewards_fixed = []

rng = np.random.default_rng(42)

for step in range(n_steps):
    # 1. RL Bandit action
    action_idx = bandit.select_action()
    action_name = ROUTING_ACTIONS[action_idx]
    reward_b = simulate_environment_step(action_name, rng=rng)
    bandit.update(action_idx, reward_b)
    rewards_bandit.append(reward_b)
    
    # 2. Random baseline
    rand_act = rng.choice(ROUTING_ACTIONS)
    rewards_random.append(simulate_environment_step(rand_act, rng=rng))
    
    # 3. Fixed baseline (clarify_user)
    rewards_fixed.append(simulate_environment_step('clarify_user', rng=rng))

cum_bandit = np.cumsum(rewards_bandit)
cum_random = np.cumsum(rewards_random)
cum_fixed = np.cumsum(rewards_fixed)

regret_bandit = np.cumsum(OPTIMAL_REWARD - np.array(rewards_bandit))
regret_random = np.cumsum(OPTIMAL_REWARD - np.array(rewards_random))

## 2. Cumulative Reward & Regret Curves (CLO05)

Plotting performance over 1,000 learning steps.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(cum_bandit, label='RL Bandit (ε-Greedy)', color='mediumseagreen', linewidth=2)
ax[0].plot(cum_random, label='Random Routing Baseline', color='gray', linestyle='--')
ax[0].plot(cum_fixed, label='Fixed Baseline (Clarify Always)', color='crimson', linestyle=':')
ax[0].set_title('Cumulative Reward over Time')
ax[0].set_xlabel('Steps')
ax[0].set_ylabel('Total Accumulated Reward')
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].plot(regret_bandit, label='RL Bandit Regret', color='mediumseagreen', linewidth=2)
ax[1].plot(regret_random, label='Random Baseline Regret', color='gray', linestyle='--')
ax[1].set_title('Cumulative Regret over Time (Lower is Better)')
ax[1].set_xlabel('Steps')
ax[1].set_ylabel('Regret (Optimal vs Actual)')
ax[1].legend()
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Paradigm Comparison: Supervised vs Unsupervised vs RL

| Paradigm | Components | Primary Role in M.O.N.D.A.Y. | Primary Metric |
|---|---|---|---|
| **Supervised Learning** | 2.1 Intent Classifier, 2.2 Task ETA | Intent routing & execution delay estimation | Accuracy, F1, MAE, R² |
| **Unsupervised Learning** | 2.3 Memory Clustering, 2.4 Assoc Mining | Topic organization & repetitive workflow discovery | Silhouette Score, Support, Lift |
| **Reinforcement Learning** | 2.5 RL Router | Dynamic online policy optimization under uncertainty | Cumulative Reward, Regret |